In [36]:
import numpy as np
from hmmlearn import hmm
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import quad

In [ ]:
# df1 =  pd.read_csv('I-80-Main-Data/trajectories_0400_0415.txt', header =None, delim_whitespace=True)  # NGSIM
# df2 =  pd.read_csv('trajectories_0500_0515.txt', header =None, delim_whitespace=True)  # NGSIM
df3 =  pd.read_csv('trajectories_0515_0530.txt', header =None, delim_whitespace=True)  # NGSIM
# df4 =  pd.read_csv('trajectories_0750am_0805am.txt', header =None, delim_whitespace=True)  # NGSIM
# df5 =  pd.read_csv('trajectories_0805am_0820am.txt', header =None, delim_whitespace=True)  # NGSIM
# df6 =  pd.read_csv('trajectories_0820am_0835am.txt', header =None, delim_whitespace=True)  # NGSIM
data = df3
# all_data['Acceleration_x'] = all_data['Acceleration_x'] * 10
# print(len(data))

# Define column names as a list
column_names = ['VehicleID', 'FrameID', 'TotalFrames', 'GlobalTime', 'LocalX', 'LocalY', 'GlobalX', 'GlobalY', 'Length', 'Width', 'Class', 'Velocity', 'Acceleration', 'LaneID', 'PrecedingVehicle', 'FollowingVehicle', 'Spacing', 'TimeHeadway']

# Set the column names of the DataFrame
data.columns = column_names
data = data.apply(pd.to_numeric, errors='coerce')

Car_data = data[data['Class'] == 2]

# Merge the dataset with itself based on the preceding vehicle ID and timestamp
# Find the minimum length of the two DataFrames
df1_01 = pd.DataFrame(Car_data)
df1_02 = pd.DataFrame(Car_data)
data1 = pd.merge(df1_01, df1_02[['VehicleID','FrameID','Velocity', 'Acceleration', 'Length', 'Spacing', 'TimeHeadway']], left_on = ['PrecedingVehicle','FrameID'], right_on = ['VehicleID','FrameID'])

# Change feet to meters
# Change feet to meters
data1['Velocity_x'] = data1['Velocity_x'] * 0.3048
data1['Spacing_x'] = data1['Spacing_x'] * 0.3048
data1['Length_x'] = data1['Length_x'] * 0.3048

data1['Velocity_y'] = data1['Velocity_y'] * 0.3048
data1['Spacing_y'] = data1['Spacing_y'] * 0.3048
data1['Length_y'] = data1['Length_y'] * 0.3048
print(len(Car_data))

1696031


In [38]:
# # Smoothing data using Kalman Filter
class KalmanFilter:
    def __init__(self, F, H, Q, R, P, x0):
        self.F = F  # State transition matrix
        self.H = H  # Observation matrix
        self.Q = Q  # Process noise covariance
        self.R = R  # Observation noise covariance
        self.P = P  # State covariance matrix
        self.x = x0  # Initial state

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.x

    def update(self, z):
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(self.P.shape[0]) - K @ self.H) @ self.P

# Initialize the Kalman Filter
F = np.array([[1.]])  # State transition matrix
H = np.array([[1.]])  # Observation matrix
Q = np.array([[0.01]])  # Process noise covariance
R = np.array([[1.0]])  # Observation noise covariance
P = np.array([[10.]])  # Initial state covariance matrix
x0 = np.array([0.])    # Initial state

kf_v = KalmanFilter(F, H, Q, R, P, x0)
kf_s = KalmanFilter(F, H, Q, R, P, x0)

# Smoothing data 01
ID_list = data1['VehicleID_x'].unique()
for i in ID_list:
    ID_tra = data1[data1['VehicleID_x'] == i]
    velocity = ID_tra['Velocity_x'].values
    spacing = ID_tra['Spacing_x'].values
    # print(velocity)
    # Apply the Kalman Filter
    smooth_data_v = np.zeros_like(velocity)
    smooth_data_s = np.zeros_like(spacing)
    for j in range(velocity.shape[0]):
        kf_v.predict()
        kf_v.update(velocity[j])
        smooth_data_v[j] = kf_v.x[0]

    for k in range(spacing.shape[0]):
        kf_s.update(spacing[k])
        smooth_data_s[k] = kf_s.x[0]

    data1.loc[data1['VehicleID_x'] == i,'Velocity_x'] = smooth_data_v
    acceleration = (smooth_data_v[1:] - smooth_data_v[:-1])
    acceleration = np.append(acceleration, 0) # add the last element as 0 最后一个加速度计算不到 设置为0
    data1.loc[data1['VehicleID_x'] == i,'Acceleration_x'] = acceleration
    data1.loc[data1['VehicleID_x'] == i,'Spacing_x'] = smooth_data_s

In [39]:
# Have the proceeding vehicle
data1 = data1[data1['LaneID'] < 6]

# have preceding vehicle
remove1 = data1[data1['PrecedingVehicle'] < 1]['VehicleID_x'].unique()
data1 = data1[~data1['VehicleID_x'].isin(remove1)]

# delete 9999.0
remove2 = data1[data1['TimeHeadway_x'] > 100]['VehicleID_x'].unique()
data1 = data1[~data1['VehicleID_x'].isin(remove2)]

# no congestion
# Filter out vehicles with speeds lower than 5 m/s
# Find the vehicle IDs with speed less than 5 at some timestamps
remove3 = data1[data1['Velocity_x'] < 5]['VehicleID_x'].unique()
data1 = data1[~data1['VehicleID_x'].isin(remove3)]

remove4 = data1[data1['Spacing_x'] < 7]['VehicleID_x'].unique()
data1 = data1[~data1['VehicleID_x'].isin(remove4)]

# # select the vehicles that have more than 1 min
# Group the data by 'vehicle_id' and calculate the duration for each vehicle
duration = data1.groupby('VehicleID_x')['FrameID'].agg(lambda x: x.max() - x.min())
remove5 = duration[duration < 500].index
data1 = data1[~data1['VehicleID_x'].isin(remove5)]

data1['delta_v'] = data1['Velocity_y'] - data1['Velocity_x']
data1['distance'] = data1['Spacing_x'] - 0.5 * data1['Length_x'] - 0.5 * data1['Length_y']

In [ ]:
# save the dataset - original - without congestion
data1.to_csv('Car80_S3.csv', index=False, sep=',')